In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")

model = init_chat_model(
    model='mistral-large-latest'
)


In [3]:
import requests, pathlib

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

File downloaded and saved as Chinook.db


In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [5]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(
    db = db,
    llm = model
)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")


sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [6]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt
)

In [8]:
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which genre on average has the longest tracks?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (g4jSelPPO)
 Call ID: g4jSelPPO
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (iYibIUJtw)
 Call ID: iYibIUJtw
  Args:
    table_names: Genre, Track
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre table:
GenreId	Name
1	Rock
2	Jazz
3	Metal
*/


CREATE TABLE "Track" (
	"TrackId"

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"sql_db_query": True},
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
)

In [16]:
question = "Which genre on average has the longest tracks?"
config = {"configurable": {"thread_id": "1"}}

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    config,
    stream_mode="values",
):
    if "__interrupt__" in step:
        print("----------------------INTERRUPTED:----------------------")
        print("INTERRUPTED:")
        print("----------------------INTERRUPTED:----------------------")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print("----------------------INTERRUPTED:----------------------")
            print(request["description"])
            print("----------------------INTERRUPTED:----------------------")
    elif "messages" in step:
        step["messages"][-1].pretty_print()
    else:
        pass

================================ Human Message =================================

Which genre on average has the longest tracks?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (DKf5N2opc)
 Call ID: DKf5N2opc
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (P8J4Wkl53)
 Call ID: P8J4Wkl53
  Args:
    table_names: Genre, Track
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre table:
GenreId	Name
1	Rock
2	Jazz
3	Metal
*/


CREATE TABLE "Track" (
	"TrackId"

In [5]:
from pydantic import BaseModel, Field


class OutputSchema(BaseModel):
    answer: str = Field(..., description="answer for the question")
    confident: float = Field(..., description="from 0 to 1 how confident about the answer")


In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model = "mistral-large-latest",
    response_format = OutputSchema
)


In [7]:
question = "what is the capital of sri lanka"

answer = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

In [8]:
print(answer)

{'messages': [HumanMessage(content='what is the capital of sri lanka', additional_kwargs={}, response_metadata={}, id='e6e2c9d6-d770-4b25-a1de-6298e096d0fa'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nuwyhqq7X', 'function': {'name': 'OutputSchema', 'arguments': '{"answer": "The capital of Sri Lanka is **Sri Jayawardenepura Kotte** (administrative capital) and **Colombo** (commercial capital).", "confident": 1}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 103, 'total_tokens': 153, 'completion_tokens': 50, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-large-latest', 'model': 'mistral-large-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019d0b68-cafd-7fb3-ab15-bf41068c7a08-0', tool_calls=[{'name': 'OutputSchema', 'args': {'answer': 'The capital of Sri Lanka is **Sri Jayawardenepura Kotte** (administrative capital) and **Colombo** (commercial capital).', 'confident': 1}, 'id': 'nuw

In [ ]:
print(answer['structured_response'])

answer='The capital of Sri Lanka is **Sri Jayawardenepura Kotte** (administrative capital) and **Colombo** (commercial capital).' confident=1.0


In [11]:
print(answer['structured_response'].answer)

The capital of Sri Lanka is **Sri Jayawardenepura Kotte** (administrative capital) and **Colombo** (commercial capital).


In [12]:
print(answer['structured_response'].confident)

1.0
